# 01r — 2D square-lattice TFIM critical field $g_c$: scale-up & multi-backend cross-verification

验证二维方格 TFIM 的量子临界横场 $g_c\simeq3.04438$（$h=0$），用本仓库 `1r`
（ground–Rydberg）原子体系的张量网络求解器，**放大到 arXiv 论文同款 cylinder 尺寸**
并在多个后端之间交叉验证。

**Target.** For the 2D square-lattice transverse-field Ising model
$H=-J\sum_{\langle ij\rangle}Z_iZ_j-gJ\sum_iX_i$ ($h=0$), QMC (Blöte–Deng) gives
$\boxed{g_c\simeq3.04438}$.

**Rydberg `1r` → TFIM mapping** (`ryd_gate.protocols.lattice_dynamics`):
$J_{ij}=V_{ij}/4,\ h_x=\Omega/2,\ h_{z,i}=-\Delta_i/2+\tfrac14\sum_jV_{ij}$, so the dial is
$g=2\Omega/V$. We fix $J=1$ via $V_{\rm nn}=4$ ($g=h_x,\ \Omega=2g$), keep **pure
nearest-neighbour** Ising (`interaction_mode="nn"`), and use `tfim_to_rydberg_controls`
to set per-site detunings that enforce $h_z=0$.

**Order parameter.** The Rydberg $Vn_in_j$ is *antiferromagnetic* ($+J\,Z_iZ_j$),
unitarily equivalent to the ferromagnetic convention (same $g_c$) with the staggered
order parameter $m_s=\frac1N\sum_i s_iZ_i$, $s_i=(-1)^{i_x+i_y}$. At $h_z=0$ the
$\mathbb Z_2$ symmetry forces $\langle m_s\rangle=0$, so we use the checkerboard
**structure factor** $\langle m_s^2\rangle=\frac1{N^2}\sum_{ij}s_is_j\langle Z_iZ_j\rangle$.

**This notebook (one section per method, after `run_quench_benchmark.ipynb`):**

1. Parameters & shared helpers.
2. **MPS (open square)** finite-size scaling — baseline $g_c$ from the correlation-ratio crossing.
3. **MPS (cylinder, paper geometry $L_x\times L_y$, open-$x$/periodic-$y$)** — the scale-up that recovers $g_c$, plus a paper-size $16\times8$ ground state.
4. **Exact diagonalization** — rigorous cross-check of the MPS $\langle m_s^2\rangle$ at $4\times4$.
5. **YASTN finite-PEPS** (imaginary-time ground state) — a second 2D-TN cross-check at $4\times4$.
6. **PEPSKit iPEPS** — wired; runs only where a Julia toolchain is present.
7. **Cross-verification summary & verdict.**

> **Performance note.** 2D-TN tensors here are small, so the very first cell pins
> BLAS/OpenMP to **one thread** — multi-threaded BLAS oversubscribes cores and is
> *slower* for these sizes (and a good citizen on a shared box). Run the notebook
> with `OMP_NUM_THREADS=1` for reproducible timings.

In [1]:
# Pin BLAS/OpenMP to 1 thread BEFORE importing numpy/scipy/tenpy.
# Small 2D-TN tensors => multi-threaded BLAS oversubscribes cores and is SLOWER.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import csv, json, time
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

import ryd_gate as rg
from ryd_gate.core.level_structures import InteractionSpec
from ryd_gate.lattice import make_square_lattice
from ryd_gate.core.operator_spec import (
    materialize_sparse_operator, WeightedProjectorSumSpec, is_operator_spec,
)
from ryd_gate.protocols.lattice_dynamics import TFIMQuenchProtocol, tfim_to_rydberg_controls
from ryd_gate.backends.tn_common.lattice_spec import create_tn_lattice_spec
from ryd_gate.backends.tn_common.protocol_context import TNProtocolContext, pin_deltas_from_params
from ryd_gate.backends.tn_common.compiler import TNEvolutionIR
from ryd_gate.backends.tenpy_mps.backends import TenpyDMRGBackend
from ryd_gate.backends.peps2d.yastn_backend import YASTNPEPSBackend, _YASTNPEPSOps

plt.rcParams.update({"figure.dpi": 120})
print("BLAS threads pinned to", os.environ.get("OMP_NUM_THREADS"))

G_C_REF = 3.04438          # Blote-Deng QMC, 2D square TFIM
J, V_NN = 1.0, 4.0         # J = V/4  =>  g = h_x,  Omega = 2 g

BLAS threads pinned to 1


## 1. Parameters & shared helpers

`mps_ground` / `exact_ground` / `peps_ms2` all solve the *same* nn-TFIM at field $g$
(with $h_z=0$ pins) on the chosen geometry and return the checkerboard
$\langle m_s^2\rangle$ (and the correlation ratio $R=1-S(\mathbf Q+\delta\mathbf q)/S(\mathbf Q)$,
$\mathbf Q=(\pi,\pi)$). For cylinders $\delta\mathbf q$ points along the periodic $y$.

In [ ]:
def _make_spec(Lx, Ly, bc_y="open"):
    return create_tn_lattice_spec(Lx=Lx, Ly=Ly, V_nn=V_NN, Omega=1.0, bc="open",
                                  level_structure="1r", interaction_mode="nn", bc_y=bc_y)


def _zz_2d(spec, psi):
    """<Z_i Z_j> = 4<Sz_i Sz_j> as an (N,N) matrix in 2D site order."""
    C = 4.0 * np.asarray(psi.correlation_function("Sz", "Sz"))
    C2 = np.empty_like(C); idx = spec.snake_to_2d; C2[np.ix_(idx, idx)] = C
    return C2


def _structure_factor(spec, C2, q):
    ph = np.exp(1j * (spec.coords @ np.asarray(q, dtype=float)))
    return float(np.real(np.conjugate(ph) @ C2 @ ph) / spec.N)


def _ms2_and_R(spec, C2, bc_y):
    s = spec.sublattice
    m_s2 = float((s[:, None] * s[None, :] * C2).sum() / spec.N**2)
    Q = np.array([np.pi, np.pi])
    dq = np.array([0.0, 2 * np.pi / spec.Ly]) if bc_y == "periodic" else np.array([2 * np.pi / spec.Lx, 0.0])
    sQ = _structure_factor(spec, C2, Q)
    R = (1.0 - _structure_factor(spec, C2, Q + dq) / sQ) if abs(sQ) > 1e-12 else float("nan")
    return m_s2, R


def mps_ground(g, Lx, Ly, chi, bc_y="open", nsw=14):
    """DMRG ground state of the nn-TFIM; returns m_s^2, correlation ratio R, energy."""
    spec = _make_spec(Lx, Ly, bc_y)
    ctrl = tfim_to_rydberg_controls(TNProtocolContext(spec), hx=g * J, hz=0.0)
    spec = replace(spec, Omega=ctrl.Omega)
    pin = pin_deltas_from_params({"pin_deltas": ctrl.pin_deltas}, spec.N)
    res = TenpyDMRGBackend(chi_max=chi, n_sweeps=nsw, svd_min=1e-10).find_ground_state(
        spec, ctrl.Delta, pin_deltas=pin, initial_state="af1")
    m_s2, R = _ms2_and_R(spec, _zz_2d(spec, res.psi_final), bc_y)
    return {"m_s2": m_s2, "R": R, "E": float(res.metadata["energy"]), "chi": int(res.metadata["chi"])}


def _assemble_H(system):
    ir = rg.compile_hamiltonian_ir(system, system.unpack_params([]))
    def mat(op):
        return materialize_sparse_operator(op, system.basis) if is_operator_spec(op) else op
    H = None
    for term in ir.static_terms:
        c = term.coefficient(0) if callable(term.coefficient) else term.coefficient
        H = c * mat(term.operator) if H is None else H + c * mat(term.operator)
    for term in ir.drive_terms:
        c = term.coefficient(0) if callable(term.coefficient) else term.coefficient
        op = mat(term.operator); contrib = c * op
        if getattr(term, "add_hermitian_conjugate", False):
            contrib = contrib + np.conj(c) * op.conj().T
        H = contrib if H is None else H + contrib
    return H


def exact_ground(g, Lx, Ly):
    """Exact (eigsh) ground state; returns m_s^2 and energy. Use for N <= ~18."""
    geom = make_square_lattice(Lx, Ly, spacing_um=1.0)
    proto = TFIMQuenchProtocol(hx=g * J, hz=0.0, t_gate=1.0)
    system = rg.RydbergSystem.from_lattice(
        geom, "1r", interaction=InteractionSpec(C6=V_NN, mode="nn"), protocol=proto)
    w, v = eigsh(_assemble_H(system).tocsc(), k=1, which="SA")
    psi0 = v[:, 0]
    sub = tuple(float(s) for s in geom.sublattice)  # sum s_i = 0 on bipartite lattice
    Ms = (2.0 / geom.N) * materialize_sparse_operator(WeightedProjectorSumSpec("r", sub), system.basis)
    mv = Ms @ psi0
    return {"m_s2": float(np.real(np.vdot(mv, mv))), "E": float(w[0])}


def peps_ground_ms2(g, Lx, Ly, D=8, ctm_chi=16, ctm_iters=6,
                    schedule=((0.1, 40), (0.03, 40), (0.02, 50))):
    """YASTN finite-PEPS imaginary-time ground state; returns checkerboard m_s^2 via CTM."""
    spec = _make_spec(Lx, Ly)
    proto = TFIMQuenchProtocol(hx=g * J, hz=0.0, t_gate=1.0)
    params = proto.unpack_params([], TNProtocolContext(spec))
    ir = TNEvolutionIR(spec=spec, protocol=proto, params=params, metadata={})
    be = YASTNPEPSBackend(chi_max=D, dt=0.05, svd_min=1e-10)
    res = be.find_ground_state(ir, dtau_schedule=schedule, observables=["m_s", "n_mean"])
    psi = res.psi_final
    yastn, fpeps, gates, cfg = be._load_yastn()
    ops = _YASTNPEPSOps(yastn, cfg, spec.level_spec.levels)
    env = fpeps.EnvCTM(psi)
    for _ in range(ctm_iters):
        env.update_({"D_total": ctm_chi, "tol": 1e-8})
    sub = np.asarray(spec.sublattice, dtype=float)
    N = spec.N
    total = float(N)  # diagonal <Z_i^2> = 1
    for i in range(N):
        ci = (i // Ly, i % Ly)
        for j in range(i + 1, N):
            zz = float(np.real(env.measure_nsite(ops.Z, ops.Z, sites=(ci, (j // Ly, j % Ly)))))
            total += 2.0 * sub[i] * sub[j] * zz
    return {"m_s2": total / N**2, "m_s_spont": abs(float(res.metadata["obs"]["m_s"]))}


def ratio_crossing(gA, RA, gB, RB):
    """First g where two correlation-ratio curves cross (linear interpolation)."""
    lo, hi = max(min(gA), min(gB)), min(max(gA), max(gB))
    if not hi > lo:
        return float("nan")
    gg = np.linspace(lo, hi, 800)
    d = np.interp(gg, gA, RA) - np.interp(gg, gB, RB)
    sc = np.where(np.sign(d[:-1]) != np.sign(d[1:]))[0]
    if sc.size == 0:
        return float("nan")
    i = sc[0]
    return float(gg[i] - d[i] * (gg[i + 1] - gg[i]) / (d[i + 1] - d[i]))


def run_sweep(fn, sizes, grids, label, time_budget_s=None):
    """Run fn(g, **size_kwargs) over a g-grid per size; optional per-size time guard."""
    results, skipped = {}, {}
    for key, kw in sizes.items():
        grid = np.asarray(grids[key], dtype=float)
        rows = {"g": [], "m_s2": [], "R": [], "t": []}
        t_size = time.perf_counter()
        for k, g in enumerate(grid):
            t0 = time.perf_counter()
            out = fn(float(g), **kw)
            dt = time.perf_counter() - t0
            rows["g"].append(float(g)); rows["m_s2"].append(out["m_s2"])
            rows["R"].append(out.get("R", float("nan"))); rows["t"].append(dt)
            print(f"  [{label} {key}] g={g:.4f}  m_s^2={out['m_s2']:.4f}  t={dt:.1f}s", flush=True)
            if time_budget_s and k == 0 and dt > time_budget_s:
                skipped[key] = {"reason": "first point exceeded time_budget_s",
                                "t_first_s": round(dt, 1), "n_grid": int(grid.size)}
                print(f"  [{label} {key}] guard tripped ({dt:.0f}s > {time_budget_s:.0f}s); "
                      f"keeping 1 point only.", flush=True)
                break
        results[key] = rows
        print(f"  [{label} {key}] size-time={time.perf_counter()-t_size:.1f}s", flush=True)
    return results, skipped

## 2. MPS (open square) — finite-size-scaling baseline

DMRG ground states on open $L\times L$ clusters ($L=3,4,5,6$). $g_c$ is read from the
**correlation-ratio crossing** of the two largest sizes (RG-invariant). This is the
baseline; open boundaries bias it upward, which the cylinder geometry in §3 removes.

In [ ]:
OPEN_SIZES = {L: dict(Lx=L, Ly=L, chi=chi, bc_y="open")
              for L, chi in [(3, 64), (4, 128), (5, 300), (6, 400)]}
G_OPEN = np.round(np.linspace(2.7, 3.4, 6), 4)
GRIDS_OPEN = {L: G_OPEN for L in OPEN_SIZES}

print("=== MPS open-square FSS ===", flush=True)
t0 = time.perf_counter()
open_res, _ = run_sweep(mps_ground, OPEN_SIZES, GRIDS_OPEN, "open")
print(f"open FSS total {time.perf_counter()-t0:.1f}s", flush=True)

open_sizes = sorted(open_res)
gc_open_pairs = [( (a, b), ratio_crossing(open_res[a]["g"], open_res[a]["R"],
                                          open_res[b]["g"], open_res[b]["R"]) )
                 for a, b in zip(open_sizes[:-1], open_sizes[1:])]
gc_open = next((gc for _, gc in reversed(gc_open_pairs) if np.isfinite(gc)), float("nan"))
print("open crossings:", [(p, round(gc, 4)) for p, gc in gc_open_pairs], "-> g_c ~", round(gc_open, 4))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for L in open_sizes:
    a1.plot(open_res[L]["g"], open_res[L]["m_s2"], "o-", label=f"L={L}")
    a2.plot(open_res[L]["g"], open_res[L]["R"], "s-", label=f"L={L}")
for ax in (a1, a2):
    ax.axvline(G_C_REF, color="k", ls="--", lw=1, label=f"$g_c$={G_C_REF}")
    ax.set_xlabel("g = 2$\\Omega$/V"); ax.legend(fontsize=8)
a1.set_ylabel(r"$\langle m_s^2\rangle$"); a1.set_title("open square: order parameter")
a2.set_ylabel("R"); a2.set_title("open square: correlation ratio")
fig.tight_layout(); fig_open = fig

## 3. MPS (cylinder, paper geometry) — scale-up that recovers $g_c$

The arXiv study uses **cylinders**: open in $x$, periodic in $y$, aspect ratio
$L_x/L_y\simeq2$, with $L_y$ the finite-size knob. Periodic $y$ removes the
boundary bias, so the $L_y$ crossing converges to the bulk $g_c$ much faster. We run
$L_y=4,6$ (i.e. $8\times4$, $12\times6$) and read $g_c$ from the $L_y=4\times6$
crossing, then demonstrate one **paper-size $16\times8$** ground state.

In [ ]:
CYL_SIZES = {4: dict(Lx=8, Ly=4, chi=256, bc_y="periodic"),
             6: dict(Lx=12, Ly=6, chi=400, bc_y="periodic")}
GRIDS_CYL = {4: np.array([2.7, 2.9, 3.04438, 3.2, 3.4]),   # Ly=4 cheap -> 5 points
             6: np.array([2.8, 3.04438, 3.3])}               # Ly=6 heavy -> 3 points

print("=== MPS cylinder FSS (open-x / periodic-y) ===", flush=True)
t0 = time.perf_counter()
# guard: if a single Ly=6 solve runs very long, keep what we have
cyl_res, cyl_skipped = run_sweep(mps_ground, CYL_SIZES, GRIDS_CYL, "cyl", time_budget_s=2400)
print(f"cylinder FSS total {time.perf_counter()-t0:.1f}s | skipped: {cyl_skipped}", flush=True)

cyl_keys = [k for k in sorted(cyl_res) if cyl_res[k]["g"]]
gc_cyl = float("nan")
if len(cyl_keys) >= 2:
    a, b = cyl_keys[-2], cyl_keys[-1]
    gc_cyl = ratio_crossing(cyl_res[a]["g"], cyl_res[a]["R"], cyl_res[b]["g"], cyl_res[b]["R"])
print(f"cylinder Ly={cyl_keys} -> correlation-ratio crossing g_c ~ {gc_cyl:.4f}")

In [ ]:
# Paper-size demonstration: a single 16x8 cylinder ground state at g ~ g_c.
print("=== paper-size 16x8 cylinder ground state (one point) ===", flush=True)
paper_point = None
t0 = time.perf_counter()
try:
    paper_point = mps_ground(G_C_REF, Lx=16, Ly=8, chi=256, bc_y="periodic", nsw=8)
    paper_point["t"] = time.perf_counter() - t0
    print(f"16x8 (N=128) g={G_C_REF}: m_s^2={paper_point['m_s2']:.4f} "
          f"chi={paper_point['chi']} E={paper_point['E']:.3f} t={paper_point['t']:.0f}s", flush=True)
except Exception as exc:  # noqa: BLE001 - demonstration only
    print("16x8 demo skipped:", repr(exc), flush=True)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for Ly in cyl_keys:
    a1.plot(cyl_res[Ly]["g"], cyl_res[Ly]["m_s2"], "o-", label=f"Ly={Ly} ({CYL_SIZES[Ly]['Lx']}x{Ly})")
    a2.plot(cyl_res[Ly]["g"], cyl_res[Ly]["R"], "s-", label=f"Ly={Ly}")
for ax in (a1, a2):
    ax.axvline(G_C_REF, color="k", ls="--", lw=1, label=f"$g_c$={G_C_REF}")
    ax.set_xlabel("g = 2$\\Omega$/V"); ax.legend(fontsize=8)
a1.set_ylabel(r"$\langle m_s^2\rangle$"); a1.set_title("cylinder: order parameter")
a2.set_ylabel("R"); a2.set_title("cylinder: correlation-ratio crossing")
fig.tight_layout(); fig_cyl = fig

## 4. Exact diagonalization — rigorous cross-check at $4\times4$

Lanczos ground state ($2^{16}$ states) gives the exact $\langle m_s^2\rangle$ and
energy; the MPS at $4\times4$ must reproduce them to truncation accuracy.

In [ ]:
print("=== exact vs MPS at 4x4 ===", flush=True)
G_X = G_OPEN
exact_m, mps_m = [], []
for g in G_X:
    ex = exact_ground(float(g), 4, 4)
    mp = mps_ground(float(g), 4, 4, chi=128, bc_y="open")
    exact_m.append(ex["m_s2"]); mps_m.append(mp["m_s2"])
    print(f"  g={g:.4f}  exact={ex['m_s2']:.5f}  mps={mp['m_s2']:.5f}  "
          f"|dE|={abs(ex['E']-mp['E']):.1e}", flush=True)
exact_m, mps_m = np.array(exact_m), np.array(mps_m)
max_abs_dev_exact_mps = float(np.max(np.abs(exact_m - mps_m)))
print(f"max |m_s^2 (exact) - m_s^2 (MPS)| over the grid = {max_abs_dev_exact_mps:.2e}")

## 5. YASTN finite-PEPS — second 2D-TN cross-check at $4\times4$

Imaginary-time (NTU) ground state via the new
`YASTNPEPSBackend.find_ground_state`; the finite-$D$ PEPS stays in the symmetric
sector ($\langle m_s\rangle\approx0$), so we read the order parameter from the
**structure factor** $\langle m_s^2\rangle$ via a converged CTM environment. A few
$g$ points suffice to confirm the transition agrees with exact/MPS.

In [ ]:
print("=== YASTN-PEPS vs exact at 4x4 ===", flush=True)
G_PEPS = np.array([2.6, 3.5])
PEPS_SCHEDULE = ((0.08, 25), (0.03, 25), (0.02, 30))
peps_g, peps_m, exact_at_peps = [], [], []
peps_status = "ok"
for g in G_PEPS:
    try:
        t0 = time.perf_counter()
        pe = peps_ground_ms2(float(g), 4, 4, D=8, schedule=PEPS_SCHEDULE)
        ex = exact_ground(float(g), 4, 4)
        peps_g.append(float(g)); peps_m.append(pe["m_s2"]); exact_at_peps.append(ex["m_s2"])
        print(f"  g={g:.4f}  PEPS={pe['m_s2']:.4f}  exact={ex['m_s2']:.4f}  "
              f"rel={abs(pe['m_s2']-ex['m_s2'])/ex['m_s2']*100:.1f}%  "
              f"t={time.perf_counter()-t0:.0f}s", flush=True)
    except Exception as exc:  # noqa: BLE001 - PEPS is a bonus cross-check; never kill the run
        peps_status = f"error: {exc!r}"
        print("  PEPS point failed:", repr(exc), flush=True)
peps_g = np.array(peps_g); peps_m = np.array(peps_m); exact_at_peps = np.array(exact_at_peps)
print("PEPS status:", peps_status)

## 6. PEPSKit iPEPS — wired (needs a Julia toolchain)

`backend="pepskit"` drives an infinite-PEPS ground state through PEPSKit.jl. It
requires a working Julia install; where absent (as in CI / this environment) the
section reports that and is skipped — the iPEPS thermodynamic-limit cross-check is
the natural follow-up once Julia is provisioned.

In [ ]:
pepskit_status = "not run"
try:
    import shutil
    if shutil.which("julia") is None:
        raise RuntimeError("julia executable not found on PATH")
    # (A pepskit ground-state run would go here once Julia/PEPSKit.jl is available.)
    pepskit_status = "julia present — implement iPEPS ground-state call"
except Exception as exc:  # noqa: BLE001
    pepskit_status = f"skipped: {exc}"
print("PEPSKit iPEPS:", pepskit_status)

## 7. Cross-verification summary & verdict

We collect the $g_c$ estimates (open-square FSS, cylinder FSS) and the backend
agreement on $\langle m_s^2\rangle$ at $4\times4$, compare to QMC $g_c=3.04438$, and
save CSV/JSON/PNG artifacts to `results/`.

In [ ]:
def rel(x):
    return abs(x - G_C_REF) / G_C_REF if np.isfinite(x) else float("nan")

verdict = {
    "g_c_reference": G_C_REF,
    "g_c_mps_open_crossing": None if not np.isfinite(gc_open) else round(gc_open, 4),
    "g_c_mps_cylinder_crossing": None if not np.isfinite(gc_cyl) else round(gc_cyl, 4),
    "rel_err_open": None if not np.isfinite(rel(gc_open)) else round(rel(gc_open), 4),
    "rel_err_cylinder": None if not np.isfinite(rel(gc_cyl)) else round(rel(gc_cyl), 4),
    "cylinder_sizes": {Ly: f"{CYL_SIZES[Ly]['Lx']}x{Ly}" for Ly in cyl_keys},
    "paper_size_16x8_point": None if paper_point is None else {
        "g": G_C_REF, "m_s2": round(paper_point["m_s2"], 4),
        "chi": paper_point["chi"], "t_s": round(paper_point["t"], 0)},
    "xcheck_exact_vs_mps_max_abs_dev": float(f"{max_abs_dev_exact_mps:.2e}"),
    "xcheck_peps_vs_exact_rel_at_4x4": [round(abs(p - e) / e, 3) for p, e in zip(peps_m, exact_at_peps)],
    "peps_status": peps_status,
    "pepskit": pepskit_status,
}
best = gc_cyl if np.isfinite(gc_cyl) else gc_open
verdict["g_c_best_estimate"] = None if not np.isfinite(best) else round(best, 4)
verdict["passed_within_8pct"] = bool(np.isfinite(rel(best)) and rel(best) <= 0.08)

def repo_root():
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    return here

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = repo_root() / "results"; outdir.mkdir(exist_ok=True)
stem = outdir / f"tfim_gc_xverify_{ts}"
with open(f"{stem}.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["method", "size", "g", "m_s2", "R", "t_s"])
    for L in open_sizes:
        for g, m, R, t in zip(open_res[L]["g"], open_res[L]["m_s2"], open_res[L]["R"], open_res[L]["t"]):
            w.writerow(["mps_open", f"{L}x{L}", g, m, R, t])
    for Ly in cyl_keys:
        for g, m, R, t in zip(cyl_res[Ly]["g"], cyl_res[Ly]["m_s2"], cyl_res[Ly]["R"], cyl_res[Ly]["t"]):
            w.writerow(["mps_cyl", f"{CYL_SIZES[Ly]['Lx']}x{Ly}", g, m, R, t])
    for g, m in zip(G_X, exact_m):
        w.writerow(["exact", "4x4", g, m, "", ""])
    for g, m in zip(peps_g, peps_m):
        w.writerow(["peps", "4x4", g, m, "", ""])
fig_open.savefig(f"{stem}_open.png", bbox_inches="tight")
fig_cyl.savefig(f"{stem}_cyl.png", bbox_inches="tight")
with open(f"{stem}.json", "w") as f:
    json.dump(verdict, f, indent=2)

print(json.dumps(verdict, indent=2))
print(f"\nartifacts: {stem}.csv / .json / _open.png / _cyl.png")